# G2 집계: 기준 오답 질의의 개선 가능성

기준(EXP-014) dev-full 상위 5개 실패 177개에 대한 에이전트 두 단계 판단(`rag.g2`)과 사람 검수 기록(`data/review/g2_failures.md`)을 모은다. 검색기로 좁힐 수 있는 오답의 크기와, 사람이 메모로 짚은 요청·말 거는 문구가 실패와 관련 있는지 본다. 질의 원문은 출력하지 않는다(D-31).

In [1]:
import json
import re
from pathlib import Path

import pandas as pd

from rag.eval.metrics import gold_ranks, load_gold, read_run
from rag.g2 import final_class
from rag.index import load_cfg
from rag.search import query_list, read_texts

C:\workspace\meet-rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ROOT = Path.cwd().parent
G2 = ROOT / 'data' / 'review' / 'g2'
REVIEW = ROOT / 'data' / 'review' / 'g2_failures.md'
RUN = (ROOT / 'data' / 'runs'
       / 'snowflake-ko-fixed-512-64-hybrid-cond-filter-law-ms256-k20-dev-full.csv')
# 사람 메모가 짚은 요청·말 거는 문구(사용자 메모 2026-10-15의 예와 같은 끝맺음)
REQUEST = (r'(설명해\s*주|알려\s*주|말씀해\s*주|공유해\s*주|정리해\s*주|어떻게\s*(생각|평가|보)'
           r'|고려하고\s*계|의견이\s*있|계획(이|인)가|생각하십니까|하시나요)')

## 에이전트 분류

In [3]:
rows = pd.DataFrame([json.loads(x) for x in (G2 / 'rows.jsonl').open(encoding='utf-8')])
rows = rows.set_index('qid')
s1 = {q: json.loads((G2 / 'out1' / f'{q}.json').read_text(encoding='utf-8')) for q in rows.index}
s2 = {q: json.loads((G2 / 'out2' / f'{q}.json').read_text(encoding='utf-8')) for q in rows.index}
rows['identifiable'] = [s1[q]['identifiable'] for q in rows.index]
rows['other_answer'] = [s2[q]['other_answer'] for q in rows.index]
rows['cls'] = [final_class(s1[q], s2[q]) for q in rows.index]
print(pd.crosstab(rows['rank_type'], rows['cls'], margins=True))
print(pd.crosstab(rows['identifiable'], rows['other_answer'], margins=True))

cls        개선 가능  좁힐 수 없음  All
rank_type                     
결합 손실         12       41   53
근접             4       50   54
전 목록 실패        1       69   70
All           17      160  177
other_answer  False  True  All
identifiable                  
False            63    88  151
True             17     9   26
All              80    97  177


## 사람 검수 기록

체크 칸(동의·수정)이나 메모를 채운 항목만 센다.

In [4]:
rec = []
for b in re.split(r'\n## ', REVIEW.read_text(encoding='utf-8'))[1:]:
    q = b.split('\n', 1)[0]
    person = re.search(r'- 사람: (.*)', b).group(1)
    memo = re.search(r'- 메모:(.*?)(?:\n\n|\Z)', b, re.S).group(1).strip()
    agree, edit = re.match(r'\[(.*?)\] 동의\s+\[(.*?)\] 수정', person).groups()
    if agree.strip() or edit.strip() or memo:
        rec.append({'qid': q, 'agree': bool(agree.strip()), 'edit': bool(edit.strip()),
                    'memo': bool(memo)})
human = pd.DataFrame(rec).set_index('qid').join(rows[['cls']])
human['detail'] = [rows.loc[q, 'rank_type'] if rows.loc[q, 'cls'] == '개선 가능'
                   else s1[q]['reason'] or '다른 정답' for q in human.index]
print('기록한 항목', len(human), '/', len(rows))
print(human.groupby(['cls', 'detail'])[['agree', 'edit', 'memo']].sum())

기록한 항목 28 / 177
                  agree  edit  memo
cls     detail                     
개선 가능   결합 손실        12     0     6
        근접            4     0     0
        전 목록 실패       1     0     0
좁힐 수 없음 구분 단서 빠짐     10     0    11


## 개선 가능의 상한

개선 가능 질의를 모두 상위 5개에 넣어도 오르는 dev-full Recall@5. SPEC 채택선은 +1.0%p다.

In [5]:
cfg = load_cfg(['+exp=exp014', 'index.scope=full'])
gold = load_gold(cfg, 'dev-full')
ranks = gold_ranks(read_run(RUN), gold, 10)
n_imp = int((rows['cls'] == '개선 가능').sum())
print('기준 Recall@5', round((ranks <= 5).mean(), 4), ', 개선 가능', n_imp,
      ', 상한', f'+{n_imp / len(gold) * 100:.2f}%p')

기준 Recall@5 0.9413 , 개선 가능 17 , 상한 +0.56%p


## 요청·말 거는 문구

질의에 요청·말 거는 문구가 있는 비율을 기준 성공·실패와 실패의 분류별로 본다. 사람 메모가 개선 가능 질의에서 이 문구를 재작성 대상으로 짚었다.

In [6]:
qids, path = query_list(cfg)
texts = dict(zip(qids, read_texts(path, qids), strict=True))
q = pd.DataFrame({'req': {k: bool(re.search(REQUEST, t)) for k, t in texts.items()},
                  'len': {k: len(t) for k, t in texts.items()}})
q['miss'] = ~(ranks.reindex(q.index) <= 5)
q['cls'] = rows['cls'].reindex(q.index).fillna('성공')
print(q.groupby('cls')[['req', 'len']].agg(['mean', 'count']).round(3))
rate = q.groupby('req')['miss'].mean()
print('문구 있음 실패율', round(rate[True], 4), ', 없음', round(rate[False], 4))

           req           len      
          mean count    mean count
cls                               
개선 가능    0.471    17  72.765    17
성공       0.250  2839  71.452  2839
좁힐 수 없음  0.225   160  49.400   160
문구 있음 실패율 0.0584 , 없음 0.0588
